# 01 — Exploratory Data Analysis
**Teammate Matcher | DTSC 2302**

This notebook performs EDA on the **raw** Google Forms survey export.  
Goal: understand distributions, identify data quality issues, and motivate preprocessing decisions before the pipeline runs.

Input: `data/raw_survey_responses.csv`  
Output: observations that inform `src/preprocess.py`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── style ────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
UNCC_GREEN = "#00703C"
UNCC_GOLD  = "#B3A369"
plt.rcParams["figure.dpi"] = 120

RAW_PATH = "../data/raw_survey_responses.csv"
df_raw = pd.read_csv(RAW_PATH)
print(f"Raw shape: {df_raw.shape[0]} responses × {df_raw.shape[1]} columns")

---
## 1 · Dataset Overview

In [ ]:
df_raw.head(3)

In [ ]:
df_raw.info()

In [ ]:
# ── Missing value summary ─────────────────────────────────────────────────────
missing = df_raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(missing.to_string() if len(missing) else "  None")

---
## 2 · Course Code Variants

All responses are for DTSC 2302 but were entered inconsistently (free-text field).  
The preprocessing pipeline normalizes all to `"DTSC 2302"` and drops the column.

In [ ]:
course_col = "What class is this team assignment for?"
variants = df_raw[course_col].value_counts()
print(f"{len(variants)} unique course code entries across {df_raw.shape[0]} responses:\n")
print(variants.to_string())

---
## 3 · Respondent Demographics

In [ ]:
year_col = "What is your year in school?"
year_order = ["Freshman", "Sophomore", "Junior", "Senior", "Graduate"]
year_counts = df_raw[year_col].value_counts().reindex(year_order).dropna()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(year_counts.index, year_counts.values,
              color=UNCC_GREEN, edgecolor="white", linewidth=0.8)
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_title("Respondents by Year in School", fontweight="bold")
ax.set_ylabel("Count")
ax.set_ylim(0, year_counts.max() + 3)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
sns.despine()
plt.tight_layout()
plt.show()
print(year_counts.to_string())

---
## 4 · Schedule Availability

Days and time slots are collected as comma-separated checkbox responses.  
We expand each into a binary column in preprocessing.

In [ ]:
days_col  = "Which days are you generally available to meet with your team?"
times_col = "What time(s) of day are you available?"

DAY_LABELS  = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
TIME_LABELS = ["Morning (8 AM – 12 PM)","Afternoon (12 PM – 5 PM)",
               "Evening (5 PM – 9 PM)","Late night (9 PM+)"]

def count_checkbox(series, options):
    return pd.Series(
        {opt: series.fillna("").apply(lambda x: int(opt in x)).sum() for opt in options}
    )

day_counts  = count_checkbox(df_raw[days_col],  DAY_LABELS)
time_counts = count_checkbox(df_raw[times_col], TIME_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(day_counts.index, day_counts.values,
             color=UNCC_GREEN, edgecolor="white")
axes[0].set_title("Available Days", fontweight="bold")
axes[0].set_xlabel("Respondents")
axes[0].set_xlim(0, df_raw.shape[0] + 2)
axes[0].invert_yaxis()
for i, v in enumerate(day_counts.values):
    axes[0].text(v + 0.3, i, str(v), va="center", fontsize=9)

short_times = ["Morning","Afternoon","Evening","Late Night"]
axes[1].bar(short_times, time_counts.values,
            color=UNCC_GOLD, edgecolor="white")
axes[1].set_title("Available Time Slots", fontweight="bold")
axes[1].set_ylabel("Respondents")
for i, v in enumerate(time_counts.values):
    axes[1].text(i, v + 0.3, str(v), ha="center", fontsize=9)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ── How many days does each respondent have available? ────────────────────────
n_days = df_raw[days_col].fillna("").apply(
    lambda x: sum(1 for d in DAY_LABELS if d in x)
)
print("Days available per respondent:")
print(n_days.describe().round(2))
print(f"\nMedian: {n_days.median()}  Mode: {n_days.mode().values[0]}")

---
## 5 · Weekly Hours Commitment

In [ ]:
hours_col = "How many hours per week can you realistically dedicate to group project work?"
hours_order = ["Less than 3 hours", "3–5 hours", "6–9 hours", "10+ hours"]
hours_counts = df_raw[hours_col].value_counts().reindex(hours_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(hours_counts.index, hours_counts.values,
              color=UNCC_GREEN, edgecolor="white")
ax.bar_label(bars, padding=3)
ax.set_title("Weekly Hours Available for Group Work", fontweight="bold")
ax.set_ylabel("Respondents")
ax.tick_params(axis="x", rotation=15)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
sns.despine()
plt.tight_layout()
plt.show()

---
## 6 · Technical Skill Ratings

Eight self-rated dimensions (Likert 1–5).  
These form the `complementarity_features` used in GMM.

In [ ]:
SKILL_COLS = [
    "Python / programming",
    "Data analysis (pandas, spreadsheets, SQL)",
    "Statistics and math",
    "Data visualization (matplotlib, Tableau, etc.)",
    "Machine learning / modeling",
    "Technical writing and documentation",
    "Research and literature review",
    "Presentations and public speaking",
]
SKILL_SHORT = [
    "Python", "Data Analysis", "Statistics", "Visualization",
    "ML/Modeling", "Tech Writing", "Research", "Presenting",
]

skill_df = df_raw[SKILL_COLS].copy()
skill_df.columns = SKILL_SHORT

print("Skill rating summary (1–5 scale):")
print(skill_df.describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
skill_df.boxplot(ax=ax, vert=True, patch_artist=True,
                 boxprops=dict(facecolor=UNCC_GREEN, alpha=0.6),
                 medianprops=dict(color="white", linewidth=2),
                 whiskerprops=dict(color="#555"),
                 capprops=dict(color="#555"),
                 flierprops=dict(marker="o", markerfacecolor=UNCC_GOLD, markersize=5))
ax.set_title("Distribution of Self-Rated Technical Skills", fontweight="bold")
ax.set_ylabel("Rating (1 = Low, 5 = High)")
ax.set_ylim(0.5, 5.5)
ax.tick_params(axis="x", rotation=20)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# ── Skill correlation heatmap ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
corr = skill_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="YlGn", center=0, linewidths=0.5,
            ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Skill Rating Correlations", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 7 · Work Style Variables

In [ ]:
WORKSTYLE = {
    "What role do you naturally gravitate toward in a group?": {
        "label": "Role Preference",
        "order": [
            "I prefer to follow clear instructions",
            "I prefer to contribute as a specialist",
            "I'm comfortable in either role",
            "I prefer to lead and coordinate",
        ],
        "short": ["Follower","Specialist","Flexible","Leader"],
    },
    "How do you typically approach deadlines?": {
        "label": "Deadline Style",
        "order": [
            "I work best under pressure, closer to the deadline",
            "I work steadily throughout",
            "I finish well before the deadline",
        ],
        "short": ["Last-minute","Steady","Early"],
    },
    "How often do you prefer to check in with teammates?": {
        "label": "Check-in Frequency",
        "order": ["Only when necessary","Once a week","A few times a week","Daily"],
        "short": ["As Needed","Weekly","Few/Week","Daily"],
    },
    "When working on a group project, I prefer to...": {
        "label": "Collaboration Style",
        "order": [
            "Work independently and combine at the end",
            "A mix — divide tasks but check in regularly",
            "Collaborate closely throughout",
        ],
        "short": ["Independent","Mixed","Close Collab"],
    },
}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

colors = [UNCC_GREEN, UNCC_GOLD, "#4C8C5E", "#8B7A3D"]
for idx, (col, meta) in enumerate(WORKSTYLE.items()):
    cnts = df_raw[col].value_counts().reindex(meta["order"]).fillna(0).astype(int)
    cnts.index = meta["short"]
    bars = axes[idx].bar(cnts.index, cnts.values,
                         color=colors[idx], edgecolor="white")
    axes[idx].bar_label(bars, padding=2, fontsize=9)
    axes[idx].set_title(meta["label"], fontweight="bold")
    axes[idx].set_ylabel("Count")
    axes[idx].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    axes[idx].tick_params(axis="x", rotation=15)

sns.despine()
plt.suptitle("Work Style Distributions", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 8 · Meeting & Communication Preferences

In [ ]:
meet_col = "Do you prefer to meet in person or remotely?"
comm_col = "How do you prefer to communicate with your team? (pick primary)"

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

meet_counts = df_raw[meet_col].value_counts()
axes[0].pie(meet_counts.values, labels=meet_counts.index, autopct="%1.0f%%",
            colors=[UNCC_GREEN, UNCC_GOLD, "#888"],
            startangle=90, wedgeprops=dict(edgecolor="white", linewidth=1.5))
axes[0].set_title("Meeting Mode Preference", fontweight="bold")

comm_counts = df_raw[comm_col].value_counts()
axes[1].barh(comm_counts.index, comm_counts.values,
             color=UNCC_GREEN, edgecolor="white")
axes[1].set_title("Primary Communication Channel", fontweight="bold")
axes[1].set_xlabel("Respondents")
axes[1].invert_yaxis()
for i, v in enumerate(comm_counts.values):
    axes[1].text(v + 0.2, i, str(v), va="center", fontsize=9)

sns.despine(ax=axes[1])
plt.tight_layout()
plt.show()

---
## 9 · GPA Band Distribution

GPA is optional — "Prefer not to say" is treated as missing and imputed with the median band.

In [ ]:
gpa_col = "What is your GPA range? (Optional)"
gpa_order = ["Below 2.5", "2.5 – 3.0", "3.0 – 3.5", "3.5 – 4.0"]

raw_gpa = df_raw[gpa_col].value_counts(dropna=False)
print("Raw GPA responses:")
print(raw_gpa.to_string())

gpa_clean = df_raw[gpa_col].replace("Prefer not to say", np.nan)
gpa_counts = gpa_clean.value_counts().reindex(gpa_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(gpa_counts.index, gpa_counts.values,
              color=UNCC_GREEN, edgecolor="white")
ax.bar_label(bars, padding=3)
ax.set_title("GPA Band Distribution (excl. 'Prefer not to say')", fontweight="bold")
ax.set_ylabel("Respondents")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
missing_n = df_raw[gpa_col].isin(["Prefer not to say"]).sum() + df_raw[gpa_col].isna().sum()
ax.annotate(f"Missing/not disclosed: {missing_n}",
            xy=(0.98, 0.95), xycoords="axes fraction", ha="right",
            fontsize=9, color="gray")
sns.despine()
plt.tight_layout()
plt.show()

---
## 10 · Team Contributions (Multi-Select)

> **Note:** The survey asked respondents to select **up to 2** contributions.  
> However, many respondents selected 3–4.  
> Decision: treat as binary indicator columns ("did they select this type or not").  
> The selection limit is not enforced in the model.

In [ ]:
contrib_col = "What do you most contribute to a team? (Select up to 2)"

CONTRIB_OPTIONS = [
    "Technical execution",
    "Creative ideas",
    "Organization and planning",
    "Research and writing",
    "Keeping team morale up",
    "Quality checking / editing",
]

contrib_counts = count_checkbox(df_raw[contrib_col], CONTRIB_OPTIONS)

# How many items did each person select?
n_selected = df_raw[contrib_col].fillna("").apply(
    lambda x: sum(1 for opt in CONTRIB_OPTIONS if opt in x)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(CONTRIB_OPTIONS, contrib_counts.values,
             color=UNCC_GREEN, edgecolor="white")
axes[0].set_title("Contribution Type Frequency", fontweight="bold")
axes[0].set_xlabel("Respondents")
axes[0].invert_yaxis()
for i, v in enumerate(contrib_counts.values):
    axes[0].text(v + 0.2, i, str(v), va="center", fontsize=9)

sel_counts = n_selected.value_counts().sort_index()
axes[1].bar(sel_counts.index, sel_counts.values,
            color=UNCC_GOLD, edgecolor="white")
axes[1].axvline(x=2.5, color="red", linestyle="--", alpha=0.6, label="'Up to 2' limit")
axes[1].set_title("# of Contributions Selected per Respondent", fontweight="bold")
axes[1].set_xlabel("Selections")
axes[1].set_ylabel("Respondents")
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[1].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

sns.despine()
plt.tight_layout()
plt.show()
print(f"Respondents selecting >2: {(n_selected > 2).sum()} / {len(n_selected)}")

---
## 11 · Pain Points

In [ ]:
pain_col = "What is your biggest challenge in group projects?"
pain_counts = df_raw[pain_col].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(pain_counts.index, pain_counts.values,
        color=UNCC_GREEN, edgecolor="white")
ax.set_title("Biggest Challenge in Group Projects", fontweight="bold")
ax.set_xlabel("Respondents")
ax.invert_yaxis()
for i, v in enumerate(pain_counts.values):
    ax.text(v + 0.15, i, str(v), va="center", fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()

---
## 12 · Availability Overlap Heatmap

Visualize how many respondents overlap on each day × time slot combination.

In [ ]:
# Build day × time overlap matrix
SHORT_DAYS  = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
SHORT_TIMES = ["Morning","Afternoon","Evening","Late Night"]

overlap = np.zeros((len(TIME_LABELS), len(DAY_LABELS)), dtype=int)
for i, t in enumerate(TIME_LABELS):
    for j, d in enumerate(DAY_LABELS):
        overlap[i, j] = (
            df_raw[days_col].fillna("").str.contains(d) &
            df_raw[times_col].fillna("").str.contains(t.split(" (")[0])
        ).sum()

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    overlap, annot=True, fmt="d", cmap="YlGn",
    xticklabels=SHORT_DAYS, yticklabels=SHORT_TIMES,
    linewidths=0.5, linecolor="white",
    ax=ax, cbar_kws={"label": "# Respondents Available"}
)
ax.set_title("Respondent Availability — Day × Time Slot", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 13 · Student Profile Overview

Radar chart of mean skill ratings across the cohort.

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.path import Path

means = skill_df.mean().values
labels = SKILL_SHORT
N = len(labels)

angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
means_plot = np.concatenate([means, [means[0]]])
angles_plot = angles + [angles[0]]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.fill(angles_plot, means_plot, alpha=0.25, color=UNCC_GREEN)
ax.plot(angles_plot, means_plot, color=UNCC_GREEN, linewidth=2)
ax.set_xticks(angles)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(1, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["1","2","3","4","5"], fontsize=8, color="gray")
ax.set_title("Mean Self-Rated Skill Profile\n(all 31 respondents)",
             fontweight="bold", pad=20)
plt.tight_layout()
plt.show()
print("\nMean ratings:")
for label, mean in zip(SKILL_SHORT, means):
    print(f"  {label:<18} {mean:.2f}")

---
## 14 · EDA Summary & Preprocessing Decisions

| Observation | Preprocessing Decision |
|---|---|
| 11 course code variants, all mean DTSC 2302 | Normalize to `"DTSC 2302"`, drop column from feature sets |
| 2 empty artifact columns (Column 25) | Drop on load |
| Timestamp column present | Drop — not analytically relevant |
| GPA: 1 NaN + 1 "Prefer not to say" | Treat both as missing; impute with median band |
| Contributions: 16/31 selected >2 items | Treat as binary indicators; do not enforce limit |
| Days/times are comma-separated strings | Expand to 11 individual binary columns |
| Skills rated 1–5 (continuous), ordinals have varying ranges | Min-Max normalize all non-binary numeric features to [0, 1] |
| Afternoon is the most common availability slot | No action — captured in availability features |
| Text/iMessage dominates communication (~20 respondents) | One-hot encode; high imbalance noted for model interpretation |
| All respondents completed most fields (low missingness) | No rows dropped for >20% missing (only GPA imputed) |